# NeuroOntoGen end-to-end demo

This notebook is a reviewer-facing reproduction path for the current NeuroOntoGen MVP. It runs the deterministic SDK pipeline end to end:

`LinkML schema -> compiled SHACL -> typed ABox payload -> Turtle -> SHACL validation -> violation inspection -> bounded mock repair -> final valid Turtle`.

The repair step is intentionally provider-neutral: a deterministic mock repairer is used so the notebook can execute without external LLM credentials.


In [ ]:
from pathlib import Path
import os

from neuro_onto_gen.core.models import (
    ABoxPayload,
    ExtractedEmployee,
    ExtractedRelation,
    ExtractedSecureAsset,
)
from neuro_onto_gen.core.repair import RepairController
from neuro_onto_gen.core.serializer import serialize_abox_to_turtle
from neuro_onto_gen.core.validation import parse_shacl_violations, validate_abox_turtle
from neuro_onto_gen.schema.compiler import compile_schema

if not Path("schemas/company_schema.yaml").exists() and Path("../schemas/company_schema.yaml").exists():
    os.chdir("..")

SCHEMA_PATH = Path("schemas/company_schema.yaml")
BUILD_DIR = Path("build/notebook-schema")


## 1. Compile the LinkML schema

Compile the company LinkML schema into generated artifacts. The notebook uses the SHACL artifact as the symbolic judge for subsequent ABox graphs.


In [ ]:
artifacts = compile_schema(SCHEMA_PATH, BUILD_DIR)
artifacts


## 2. Build a typed ABox payload

Construct a small valid payload with Pydantic models. Relation endpoints are validated before RDF serialization.


In [ ]:
payload = ABoxPayload(
    employees=[
        ExtractedEmployee(emp_id="E-001", has_access_level=3),
    ],
    secure_assets=[
        ExtractedSecureAsset(asset_id="VPN", required_clearance=2),
    ],
    relations=[
        ExtractedRelation(
            subject_emp_id="E-001",
            predicate="operates",
            object_asset_id="VPN",
        ),
    ],
)
payload


## 3. Serialize the ABox to Turtle

Serialize the typed ABox into RDF/Turtle using the SDK serializer.


In [ ]:
valid_turtle = serialize_abox_to_turtle(payload)
print(valid_turtle)


## 4. Validate the Turtle graph with SHACL

Validate the Turtle graph against the generated SHACL shape. The conforming graph must pass before any repair logic is trusted.


In [ ]:
valid_report = validate_abox_turtle(valid_turtle, artifacts["shacl"])
assert valid_report.conforms is True
print(valid_report.report_text)


## 5. Show a SHACL violation

Now create a deliberately non-conforming graph that omits `requiredClearance` on a `SecureAsset`. This demonstrates that the symbolic validator catches a graph that might otherwise look plausible.


In [ ]:
invalid_turtle = """
@prefix ex: <http://example.org/company/> .

<http://example.org/company/asset/VPN> a ex:SecureAsset ;
    ex:assetId "VPN" .
"""

invalid_report = validate_abox_turtle(invalid_turtle, artifacts["shacl"])
assert invalid_report.conforms is False
violations = parse_shacl_violations(invalid_report)
assert violations
assert any("requiredClearance" in (violation.result_path or "") for violation in violations)
violations


## 6. Run bounded mock repair

Use a deterministic repairer that returns the previously validated Turtle graph. The controller still re-runs SHACL validation; the repairer never self-certifies.


In [ ]:
class DemoRepairer:
    def __init__(self, repaired_turtle: str) -> None:
        self.repaired_turtle = repaired_turtle
        self.calls = []

    def repair(self, turtle: str, violations: list, attempt_number: int) -> str:
        self.calls.append(
            {
                "attempt_number": attempt_number,
                "input_had_requiredClearance": "requiredClearance" in turtle,
                "violation_count": len(violations),
            }
        )
        return self.repaired_turtle


repairer = DemoRepairer(valid_turtle)
controller = RepairController(
    shacl_path=artifacts["shacl"],
    repairer=repairer,
    max_attempts=2,
)
repair_result = controller.repair_until_valid(invalid_turtle)

assert repair_result.succeeded is True
assert repair_result.final_report.conforms is True
assert len(repair_result.attempts) == 1
repairer.calls


## 7. Display the final valid Turtle graph

The final graph is the controller output after deterministic revalidation.


In [ ]:
print(repair_result.final_turtle)
